In [ ]:
# 1️⃣ نصب کتابخانه‌ها
!pip install sqlalchemy pandas

# 2️⃣ وارد کردن کتابخانه‌ها
import pandas as pd
from sqlalchemy import create_engine
import sqlite3

# 3️⃣ بارگذاری CSV
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")

# 4️⃣ ساخت دیتابیس SQLite و جدول SPACEXTBL
engine = create_engine('sqlite:///spacex.db')
df.to_sql("SPACEXTBL", con=engine, if_exists='replace', index=False)

# 5️⃣  اتصال با sqlite3 برای اجرای SQL با Pandas
conn = sqlite3.connect("spacex.db")


In [ ]:
#6.1 تمام سایت‌های منحصر به فرد
df1 = pd.read_sql("SELECT DISTINCT Launch_Site FROM SPACEXTBL", conn)
print(df1)
#تمام سایت‌های منحصر به فرد پرتاب

    Launch_Site
0   CCAFS LC-40
1   VAFB SLC-4E
2    KSC LC-39A
3  CCAFS SLC-40


In [ ]:
#6.2 رکوردهایی که Launch_Site با 'CCA' شروع می‌شوند
df2 = pd.read_sql("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%'", conn)
print(df2.head())
#سایت‌هایی که با 'CCA' شروع می‌شوند

         Date Time (UTC) Booster_Version  Launch_Site  \
0  2010-06-04   18:45:00  F9 v1.0  B0003  CCAFS LC-40   
1  2010-12-08   15:43:00  F9 v1.0  B0004  CCAFS LC-40   
2  2012-05-22    7:44:00  F9 v1.0  B0005  CCAFS LC-40   
3  2012-10-08    0:35:00  F9 v1.0  B0006  CCAFS LC-40   
4  2013-03-01   15:10:00  F9 v1.0  B0007  CCAFS LC-40   

                                             Payload  PAYLOAD_MASS__KG_  \
0               Dragon Spacecraft Qualification Unit                  0   
1  Dragon demo flight C1, two CubeSats, barrel of...                  0   
2                              Dragon demo flight C2                525   
3                                       SpaceX CRS-1                500   
4                                       SpaceX CRS-2                677   

       Orbit         Customer Mission_Outcome      Landing_Outcome  
0        LEO           SpaceX         Success  Failure (parachute)  
1  LEO (ISS)  NASA (COTS) NRO         Success  Failure (parachute)  

In [ ]:
#6.3 مجموع جرم محموله برای مشتری NASA (CRS)
df3 = pd.read_sql("SELECT SUM(PAYLOAD_MASS__KG_) AS total_mass_nasa FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'", conn)
print(df3)
#مجموع جرم محموله برای NASA (CRS)

   total_mass_nasa
0            45596


In [ ]:
#6.4 میانگین جرم محموله برای بوستر F9 v1.1
df4 = pd.read_sql("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_mass_f9v1_1 FROM SPACEXTBL WHERE Booster_Version = 'F9 v1.1'", conn)
print(df4)
#میانگین جرم محموله برای F9 v1.1

   avg_mass_f9v1_1
0           2928.4


In [ ]:
#6.5 اولین موفقیت فرود روی Ground Pad
df5 = pd.read_sql("SELECT MIN(Date) AS first_success_ground_pad FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'", conn)
print(df5)
#اولین موفقیت فرود روی Ground Pad

  first_success_ground_pad
0               2015-12-22


In [ ]:
#6.6 بوسترهایی که موفق روی Drone Ship شدند و جرم بین 4000 تا 6000
df6 = pd.read_sql("""
SELECT Booster_Version
FROM SPACEXTBL
WHERE Landing_Outcome = 'Success (drone ship)'
  AND PAYLOAD_MASS__KG_ > 4000
  AND PAYLOAD_MASS__KG_ < 6000
""", conn)
print(df6)
#بوسترهایی که موفق روی Drone Ship شدند و جرم بین 4000 تا 6000:

  Booster_Version
0     F9 FT B1022
1     F9 FT B1026
2  F9 FT  B1021.2
3  F9 FT  B1031.2


In [ ]:
#6.7 شمارش انواع Mission_Outcome
df7 = pd.read_sql("SELECT Mission_Outcome, COUNT(*) AS count FROM SPACEXTBL GROUP BY Mission_Outcome", conn)
print(df7)
#شمارش انواع Mission_Outcome:

                    Mission_Outcome  count
0               Failure (in flight)      1
1                           Success     98
2                          Success       1
3  Success (payload status unclear)      1


In [ ]:
#6.8 بیشترین جرم محموله و بوسترهایی که آن را حمل کرده‌اند
df8 = pd.read_sql("""
SELECT Booster_Version
FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)
""", conn)
print(df8)
#بوسترهایی که بیشترین جرم محموله را حمل کرده‌اند

   Booster_Version
0    F9 B5 B1048.4
1    F9 B5 B1049.4
2    F9 B5 B1051.3
3    F9 B5 B1056.4
4    F9 B5 B1048.5
5    F9 B5 B1051.4
6    F9 B5 B1049.5
7   F9 B5 B1060.2 
8   F9 B5 B1058.3 
9    F9 B5 B1051.6
10   F9 B5 B1060.3
11  F9 B5 B1049.7 


In [ ]:
#6.9 رکوردهای سال 2015 با فرود ناموفق روی Drone Ship و شماره ماه
df9 = pd.read_sql("""
SELECT SUBSTR(Date,6,2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTBL
WHERE Date LIKE '2015%' AND Landing_Outcome = 'Failure (drone ship)'
""", conn)
print(df9.head())
#رکوردهای سال 2015 با فرود ناموفق روی Drone Ship و شماره ماه:")

  Month       Landing_Outcome Booster_Version  Launch_Site
0    01  Failure (drone ship)   F9 v1.1 B1012  CCAFS LC-40
1    04  Failure (drone ship)   F9 v1.1 B1015  CCAFS LC-40


In [ ]:
#6.10 فیلتر رکوردها بین دو تاریخ و شمارش Landing_Outcome با رتبه‌بندی
df10 = pd.read_sql("""
SELECT Landing_Outcome, COUNT(*) AS Count
FROM SPACEXTBL
WHERE Date >= '2010-06-04' AND Date <= '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Count DESC
""", conn)
print(df10)
#شمارش Landing_Outcome بین دو تاریخ

          Landing_Outcome  Count
0              No attempt     10
1    Success (drone ship)      5
2    Failure (drone ship)      5
3    Success (ground pad)      3
4      Controlled (ocean)      3
5    Uncontrolled (ocean)      2
6     Failure (parachute)      2
7  Precluded (drone ship)      1
